# Recursion, Backtracking, Dynamic Programming and Greedy

Notebook for **Chapter 11** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Recursion and Memoization

In [ ]:
import functools

def fact(n): return 1 if n <= 1 else n * fact(n - 1)

calls = 0
def fibn(n):                      # plain, no memory
    global calls; calls += 1
    return n if n < 2 else fibn(n-1) + fibn(n-2)

@functools.lru_cache(None)
def fib(n): return n if n < 2 else fib(n-1) + fib(n-2)

print(fact(5))
print(fibn(20), calls)
print(fib(50), fib.cache_info().misses)

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def fact(n): return n * fact(n - 1)
fact(5)

RecursionError: maximum recursion depth exceeded


## 11.2 Backtracking: Subsets, Permutations, N-Queens

In [ ]:
def subsets(a):
    res = []
    def go(i, cur):
        if i == len(a): res.append(cur[:]); return    # copy, not cur itself
        go(i + 1, cur)                 # skip a[i]
        cur.append(a[i]); go(i + 1, cur); cur.pop()   # take a[i], then undo
    go(0, []); return res

def perms(a):
    res = []
    def go(cur, used):
        if len(cur) == len(a): res.append(cur[:]); return
        for i in range(len(a)):
            if not used[i]:
                used[i] = True; cur.append(a[i])
                go(cur, used)
                cur.pop(); used[i] = False
    go([], [False] * len(a)); return res

def nq(n):
    sols = []; cols = []          # cols[r] = column of the queen in row r
    def ok(r, c):
        return all(c != cc and abs(c - cc) != r - rr for rr, cc in enumerate(cols))
    def go(r):
        if r == n: sols.append(cols[:]); return
        for c in range(n):
            if ok(r, c): cols.append(c); go(r + 1); cols.pop()
    go(0); return sols

In [ ]:
if i == len(a): res.append(cur); return     # no copy

[[], [], [], [], [], [], [], []]


## 11.3 Dynamic Programming: The Recipe and 1D Problems

In [ ]:
def climb(n):                       # ways to climb n stairs with steps of 1 or 2
    a, b = 1, 1
    for _ in range(n - 1): a, b = b, a + b
    return b

def coins(cs, t):                   # fewest coins to make each amount 0..t
    dp = [0] + [float('inf')] * t
    for x in range(1, t + 1):
        for c in cs:
            if c <= x: dp[x] = min(dp[x], dp[x - c] + 1)
    return dp

def rob(a):                         # max sum, no two neighbours
    p2 = p1 = 0
    for x in a: p2, p1 = p1, max(p1, p2 + x)
    return p1

In [ ]:
dp = [0] * (t + 1)      # wrong: every amount looks free

## 11.4 Knapsack and Longest Increasing Subsequence

In [ ]:
import bisect
def knap(items, W):
    dp = [0] * (W + 1)
    for w, v in items:
        for c in range(W, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp

def lis_dp(a):
    dp = [1] * len(a)
    for i in range(len(a)):
        for j in range(i):
            if a[j] < a[i]: dp[i] = max(dp[i], dp[j] + 1)
    return dp

def lis(a):
    t = []
    for x in a:
        i = bisect.bisect_left(t, x)
        if i == len(t): t.append(x)
        else: t[i] = x
    return len(t)

print(knap([(1,1),(3,4),(4,5),(5,7)], 7))
print(lis_dp([10,9,2,5,3,7,101,18]), lis([10,9,2,5,3,7,101,18]))

In [ ]:
for c in range(w, W + 1):               # wrong direction for 0/1 knapsack
    dp[c] = max(dp[c], dp[c - w] + v)

## 11.5 Two-Dimensional DP: LCS, Edit Distance, Grids and Trees

In [ ]:
def lcs(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp

def edit(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(len(a)+1): dp[i][0] = i
    for j in range(len(b)+1): dp[0][j] = j
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1] if a[i-1] == b[j-1] else \
                       1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
dp[i][j] = dp[i-1][j-1] if a[i] == b[j] else ...    # wrong index

IndexError: string index out of range


In [ ]:
def grid(g):
    R, C = len(g), len(g[0]); dp = [[0]*C for _ in range(R)]
    for i in range(R):
        for j in range(C):
            best = 0 if i == j == 0 else min(dp[i-1][j] if i else 9e9, dp[i][j-1] if j else 9e9)
            dp[i][j] = g[i][j] + best
    return dp

In [ ]:
def rob3(t):                          # t = (value, left, right)
    def go(n):
        if not n: return 0, 0
        l, r = go(n[1]), go(n[2])
        return n[0] + l[1] + r[1], max(l) + max(r)
    return max(go(t))
print(rob3((3, (2, None, (3, None, None)), (3, None, (1, None, None)))))   # 7

## 11.6 Greedy Algorithms and Intervals

In [ ]:
def intervals_merge(a):
    a = sorted(a); out = [a[0][:]]
    for s, e in a[1:]:
        if s <= out[-1][1]: out[-1][1] = max(out[-1][1], e)
        else: out.append([s, e])
    return out

def max_meet(a):                      # most non-overlapping meetings
    a = sorted(a, key=lambda x: x[1]); end = -1; ch = []
    for s, e in a:
        if s >= end: ch.append((s, e)); end = e
    return ch

def can_jump(a):
    far = 0
    for i, x in enumerate(a):
        if i > far: return False
        far = max(far, i + x)
    return True

In [ ]:
out[-1][1] = e        # wrong: [1,10] then [2,3] would shrink the end to 3

## 11.7 Bit Manipulation and Number Theory Basics

In [ ]:
n = 44
print(bin(n), n & 1, n >> 2, bin(n & (n-1)), bin(n & -n), n & (n-1) == 0, bin(1 << 5))
print(bin(44).count('1'), [x for x in range(1, 17) if x & (x-1) == 0])
def single(a):
    r = 0
    for x in a: r ^= x
    return r
print(single([4, 1, 2, 1, 2]))

In [ ]:
r = 0
for x in [4, 1, 2, 1, 2]: r += x      # gives 10, not 4

In [ ]:
from math import gcd, isqrt
print(gcd(48, 18), 48*18 // gcd(48, 18))
def sieve(n):
    p = [True]*(n+1); p[0:2] = [False, False]
    for i in range(2, isqrt(n)+1):
        if p[i]:
            for j in range(i*i, n+1, i): p[j] = False
    return [i for i, x in enumerate(p) if x]
print(sieve(50))
print(pow(2, 10, 1000), pow(3, 200, 1_000_000_007))

6 144
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
24 136318165


## 11.8 String Matching: KMP, the Z-Array and Rabin-Karp

In [ ]:
def build_lps(p):
    lps = [0] * len(p)
    k = 0                                   # length of the current border
    for i in range(1, len(p)):
        while k > 0 and p[i] != p[k]:
            k = lps[k - 1]                  # fall back to the next shorter border
        if p[i] == p[k]:
            k += 1
        lps[i] = k
    return lps

def kmp_search(text, p):
    lps = build_lps(p); k = 0; hits = []; steps = 0
    for i, ch in enumerate(text):
        while k > 0 and ch != p[k]:
            k = lps[k - 1]; steps += 1
        steps += 1
        if ch == p[k]:
            k += 1
        if k == len(p):
            hits.append(i - k + 1)
            k = lps[k - 1]                  # keep going, allow overlapping matches
    return hits, steps

def naive_count(text, p):                   # number of letter comparisons
    c = 0
    for s in range(len(text) - len(p) + 1):
        for j in range(len(p)):
            c += 1
            if text[s + j] != p[j]: break
    return c

print(build_lps("ababaca"))
print(kmp_search("abababacaba", "ababaca"))
print(kmp_search("aaaaa", "aa"))             # overlapping matches
t, p = "a" * 1000, "a" * 20 + "b"
print("naive comparisons:", naive_count(t, p), " KMP steps:", kmp_search(t, p)[1])

[0, 0, 1, 2, 3, 0, 1]
([2], 12)
([0, 1, 2, 3], 5)
naive comparisons: 20580  KMP steps: 1980


In [ ]:
def z_array(s):
    n = len(s); z = [0] * n; z[0] = n; l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])       # reuse what the window already knows
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def z_search(text, p):
    joined = p + "#" + text
    z = z_array(joined)
    return [i - len(p) - 1 for i in range(len(p) + 1, len(joined)) if z[i] == len(p)]

print(z_array("aabxaab"))
print(z_search("xaabaab", "aab"))
print(z_search("abababacaba", "aba"))

[7, 1, 0, 0, 3, 1, 0]
[1, 4]
[0, 2, 4, 8]


In [ ]:
def rabin_karp(text, p, B=256, M=1_000_000_007):
    n, m = len(text), len(p)
    if m > n: return [], 0
    top = pow(B, m - 1, M)
    hp = ht = 0
    for i in range(m):
        hp = (hp * B + ord(p[i])) % M
        ht = (ht * B + ord(text[i])) % M
    hits, compares = [], 0
    for s in range(n - m + 1):
        if hp == ht:
            compares += 1                      # hash equal: confirm with the real strings
            if text[s:s + m] == p: hits.append(s)
        if s + m < n:
            ht = ((ht - ord(text[s]) * top) * B + ord(text[s + m])) % M
    return hits, compares

print(rabin_karp("abracadabra", "abra"))
print(rabin_karp("abracadabra", "cad"))
print(rabin_karp("abracadabra", "xyz"))
# a deliberately bad modulus shows collisions: with M = 7 many windows share a hash
print(rabin_karp("abracadabra", "abra", M=7))

([0, 7], 2)
([4], 1)
([], 0)
([0, 7], 5)


## 11.9 DP Patterns Beyond the Recipe: Counting, States, Bitmasks

The recipe from 11.3 (define the state, write the recurrence, fix the order, find the base case) covers most problems. Four extra patterns appear again and again. For each one you will see a hand trace and verified code.

In [ ]:
def combinations(coins, amount):
    dp = [1] + [0] * amount
    for c in coins:                           # coins outer: each coin used in a fixed order
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def sequences(coins, amount):
    dp = [1] + [0] * amount
    for a in range(1, amount + 1):            # amounts outer: any order allowed
        for c in coins:
            if a >= c: dp[a] += dp[a - c]
    return dp[amount]

def fewest(coins, amount):
    INF = 10**9; dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if a >= c: dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] < INF else -1

print(combinations([1, 2, 5], 5), sequences([1, 2, 5], 5))
print(combinations([1, 2, 5], 11), sequences([1, 2, 5], 11))
print(fewest([1, 2, 5], 11), fewest([2], 3))

4 9
11 218
3 -1


In [ ]:
def max_profit_cooldown(prices):
    hold, sold, rest = -prices[0], 0, 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)

print(max_profit_cooldown([1, 2, 3, 0, 2]))
print(max_profit_cooldown([1]))
print(max_profit_cooldown([6, 1, 3, 2, 4, 7]))

3
0
6


In [ ]:
from itertools import permutations
D = [[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]]
n = len(D); INF = float("inf")
dp = [[INF] * n for _ in range(1 << n)]
dp[1][0] = 0                                       # visited only city 0, standing at 0
for mask in range(1 << n):
    for last in range(n):
        if dp[mask][last] == INF: continue
        for nxt in range(n):
            if not mask & (1 << nxt):               # city nxt not yet visited
                nm = mask | (1 << nxt)
                dp[nm][nxt] = min(dp[nm][nxt], dp[mask][last] + D[last][nxt])
full = (1 << n) - 1
best = min(dp[full][last] + D[last][0] for last in range(1, n))
brute = min(D[0][a] + D[a][b] + D[b][c] + D[c][0] for a, b, c in permutations([1, 2, 3]))
print("DP:", best, " brute force:", brute)

DP: 80  brute force: 80


## 11.10 Seeing DP: Call Counts and Tables

In [ ]:
def naive(n, c):
    c[0] += 1
    return n if n < 2 else naive(n - 1, c) + naive(n - 2, c)

def memo(n, c, m):
    c[0] += 1
    if n < 2: return n
    if n not in m: m[n] = memo(n - 1, c, m) + memo(n - 2, c, m)
    return m[n]

for n in (10, 20, 30):
    a, b = [0], [0]
    naive(n, a); memo(n, b, {})
    print(n, "naive calls:", a[0], " memo calls:", b[0], " 2n-1 =", 2 * n - 1)

10 naive calls: 177  memo calls: 19  2n-1 = 19
20 naive calls: 21891  memo calls: 39  2n-1 = 39
30 naive calls: 2692537  memo calls: 59  2n-1 = 59


In [ ]:
from itertools import combinations
items = [(1, 1), (3, 4), (4, 5), (5, 7)]
best = (0, ())
for r in range(len(items) + 1):
    for subset in combinations(range(len(items)), r):
        w = sum(items[i][0] for i in subset); v = sum(items[i][1] for i in subset)
        if w <= 7 and v > best[0]: best = (v, tuple(i + 1 for i in subset))
print("best value and items:", best)

best value and items: (9, (2, 3))


## Chapter recap

## Practice set

**1.** What is `fact(4)`? **Answer:** 24.

In [ ]:
def subsets(a):
    res = []
    def go(i, cur):
        if i == len(a): res.append(cur); return
        go(i+1, cur); cur.append(a[i]); go(i+1, cur); cur.pop()
    go(0, []); return res

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch11.html